<a href="https://colab.research.google.com/github/CactusDataMafia/Yadro_AI_schoo_sem_1_ML/blob/main/%D0%9B%D0%B0%D0%B1%D0%BE%D1%80%D0%B0%D1%82%D0%BE%D1%80%D0%BD%D0%B0%D1%8F_%D1%80%D0%B0%D0%B1%D0%BE%D1%82%D0%B0_5_1_(Logreg).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Задача

Реализовать класс `MyBinaryLogisticRegression` для работы с логистической регрессией. Обеспечить возможность использования `l1`, `l2` и `l1l2` регуляризации и реализовать слудующие методы решения оптимизационной задачи:

*   Градиентный спуск
*   Стохастический градиентный спуск
*   Метод Ньютона

Обосновать применимость/не применимость того или иного метода оптимизации в случае использованного типа регуляризации.



In [ ]:
import numpy as np
import pandas as pd

In [ ]:
class MyBinaryLogisticRegression:
    def __init__(self, penalty="l2", alpha=1.0, solver="gd", lr=0.01):
        self.penalty = penalty
        self.alpha = alpha
        self.solver = solver
        self.coefs_ = None
        self.lr = lr
        self.feature_names_in_ = None


    def fit(self, X: pd.DataFrame, y: pd.DataFrame):
      n_samples, n_features = X.shape
      X_new = np.hstack((np.ones((X.shape[0], 1)), X))
      y_new = y.values.reshape(-1)
      sigmoid = lambda x: 1 / (1 + np.exp(-x))

      self.coefs_ = np.random.normal(size=X_new.shape[1])

      if self.solver == "gd":
        for _ in range(10_000):
          lin_comb = X_new @ self.coefs_
          pred = sigmoid(lin_comb)
          error = y - pred
          grad = (X_new.T @ error) / n_samples

          if self.penalty == "l1":
            reg_add = self.alpha * np.sign(self.coefs_)
          elif self.penalty == "l2":
            reg_add = self.alpha * self.coefs_
          elif self.penalty == "l1l2":
            reg_add = self.alpha * (np.sign(self.coefs_) + self.coefs_)
          else:
            raise ValueError("Такой тип регулязирации не поддерживается")

          reg_add[0] = 0
          grad += reg_add

          self.coefs_ = self.coefs_ + self.lr * grad

          if np.linalg.norm(grad) < 1e-6:
            break

      elif self.solver == "sgd":
        for _ in range(10_000):
          i = np.random.randint(0, n_samples)
          el = X_new[i]
          y_i = y_new[i]
          pred = sigmoid(el @ self.coefs_)
          error = y_i - pred
          grad = el * error

          if self.penalty == "l1":
            reg_add = self.alpha * np.sign(self.coefs_)
          elif self.penalty == "l2":
            reg_add = self.alpha * self.coefs_
          elif self.penalty == "l1l2":
            reg_add = self.alpha * (np.sign(self.coefs_) + self.coefs_)
          else:
            raise ValueError("Такой тип регулязирации не поддерживается")

          reg_add[0] = 0
          grad += reg_add

          self.coefs_ = self.coefs_ + self.lr * grad

          if np.linalg.norm(grad) < 1e-6:
            break

      elif self.solver == "newton":
        if self.penalty == "l1" or self.penalty == "l1l2":
          raise ValueError("Такие типы регуляризации несовместыми")

        for _ in range(100):
          pred = sigmoid(X_new @ self.coefs_)
          inv_pred = 1 - pred
          sigmoid_matrix = (pred * inv_pred)
          error = y - pred
          grad = (X_new.T @ error) / n_samples
          hessian = X_new.T @ (sigmoid_matrix[:, None] * X_new) / n_samples

          if self.penalty == "l2":
            hessian += self.alpha * np.eye(hessian.shape[0])
            reg_add = self.alpha * self.coefs_
          else:
            raise ValueError("Такой тип регулязирации не поддерживается")

          hessian[0, 0] -= self.alpha
          reg_add[0] = 0
          grad += reg_add

          self.coefs_ = self.coefs_ + np.linalg.inv(hessian) @ grad

          if np.linalg.norm(grad) < 1e-6:
            break

    def predict(self, X: np.array):
      sigmoid = lambda x: 1 / (1 + np.exp(-x))
      X_new = np.hstack((np.ones((X.shape[0], 1)), X))
      dot_product = X_new @ self.coefs_
      y_pred = sigmoid(dot_product)
      return y_pred

    def score(self, X: np.array, y: np.array):
        y_pred = self.predict(X)
        y_binary = (y_pred >= 0.5).astype(int)
        y_true = y.reshape(-1)

        TP = ((y_binary == 1) & (y_true == 1)).sum()
        FP = ((y_binary == 1) & (y_true == 0)).sum()
        FN = ((y_binary == 0) & (y_true == 1)).sum()

        precision = TP / (TP + FP)
        recall = TP / (TP + FN)

        f1_score = (2 * precision * recall) / (precision + recall)
        return f1_score



Продемонстрировать применение реализованного класса на датасете про пингвинов (целевая переменная — вид пингвина). Рассмотреть все возможные варианты (регуляризация/оптимизация). Для категориального признака `island` реализовать самостоятельно преобразование `Target Encoder`, сравнить результаты классификации с `one-hot`. В качестве метрики использовать `f1-score`.

# Теоретическая часть

Пусть данные имеют вид
$$
(x_i, y_i), \quad y_i \in \{1, \ldots,M\}, \quad i \in \{1, \ldots, N\},
$$
причем первая координата набора признаков каждого объекта равна $1$.
Используя `softmax`-подход, дискриминативная модель имеет следующий вид
$$
\mathbb P(C_k|x) = \frac{\exp(\omega_k^Tx)}{\sum_i \exp(\omega_i^Tx)}.
$$
Для написания правдоподобия удобно провести `one-hot` кодирование меток класса, сопоставив каждому объекту $x_i$ вектор $\widehat y_i = (y_{11}, \ldots, y_{1M})$ длины $M$, состоящий из нулей и ровно одной единицы ($y_{iy_i} = 1$), отвечающей соответствующему классу. В этом случае правдоподобие имеет вид
$$
\mathbb P(D|\omega) = \prod_{i = 1}^{N}\prod_{j = 1}^M \mathbb P(C_j|x_i)^{y_{ij}}.
$$
Ваша задача: вывести функцию потерь, градиент и гессиан для многоклассовой логистической регрессии. Реализовать матрично. На синтетическом примере продемонстрировать работу алгоритма, построить гиперплоскости, объяснить классификацию